In [1]:
import json
import os
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

os.makedirs("/workspace/outputs/metrics", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("5_Train_ALS_Spark")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.driver.host", "notebook")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "1")
    .config("spark.executor.instances", "1")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

HDFS_GOLD = "hdfs://namenode:8020/netflix-recsys/gold"
HDFS_MODELS = "hdfs://namenode:8020/netflix-recsys/models"
HDFS_OUTPUTS = "hdfs://namenode:8020/netflix-recsys/outputs"

train = spark.read.parquet(f"{HDFS_GOLD}/als_train")
test = spark.read.parquet(f"{HDFS_GOLD}/als_test")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 08:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=32,
    maxIter=8,
    regParam=0.08,
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=False,
    seed=42
)

als_model = als.fit(train)

In [3]:
predictions = als_model.transform(test).dropna()

rmse_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

mae_evaluator = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)

metrics = {
    "model": "Spark ALS",
    "rank": 32,
    "maxIter": 8,
    "regParam": 0.08,
    "rmse": rmse,
    "mae": mae
}

metrics

{'model': 'Spark ALS',
 'rank': 32,
 'maxIter': 8,
 'regParam': 0.08,
 'rmse': 0.778016972618774,
 'mae': 0.6041632994208022}

In [4]:
with open("/workspace/outputs/metrics/als_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

In [5]:
user_recs = als_model.recommendForAllUsers(10)

user_recs.write.mode("overwrite").parquet(
    f"{HDFS_OUTPUTS}/als_top10_recommendations"
)

predictions.write.mode("overwrite").parquet(
    f"{HDFS_OUTPUTS}/als_predictions"
)

als_model.write().overwrite().save(f"{HDFS_MODELS}/als_model")